# MedGemma-4B LoRA fine-tune on caries data (A100)

Fine-tunes `google/medgemma-1.5-4b-it` on the patient-safe, occlusal-only split we built for YOLO.
Same VLM training format as `finetune_medgemma.py` — this notebook just wraps it for Colab with the right environment.

**Setup**: Runtime → Change runtime type → **A100 GPU**.
A100-40GB works with 4-bit QLoRA (this notebook's default). A100-80GB works with bf16 (edit the `--four-bit` flag below to remove it).

**Gated model**: `google/medgemma-1.5-4b-it` requires you to accept the license on Hugging Face first.
Do that once at [https://huggingface.co/google/medgemma-1.5-4b-it](https://huggingface.co/google/medgemma-1.5-4b-it), then create an HF token with **Read** access and paste it when this notebook asks.

**Expected wall time**: ~1-2 hours end-to-end (download 15 min, build data 5 min, fine-tune ~60-90 min, save & download 5 min).


## 1. Verify GPU

In [ ]:
import subprocess
print(subprocess.check_output(['nvidia-smi']).decode())


## 2. Install pinned dependencies

In [ ]:
# Multimodal SFT surface in transformers changes often; use recent stable versions.
# If a later cell throws a shape/API error, the '# >>> VERIFY' tags in
# finetune_medgemma.py are the first places to look.
!pip install -q --upgrade \
    'transformers>=4.45' \
    'peft>=0.12' \
    'accelerate>=0.34' \
    'bitsandbytes>=0.43' \
    'datasets>=2.20' \
    'pillow' 'pyyaml'


## 3. Log in to Hugging Face (for the gated MedGemma weights)

In [ ]:
import os
from huggingface_hub import login, whoami, hf_hub_download
from huggingface_hub.utils import GatedRepoError, RepositoryNotFoundError
import sys

# Set HF_TOKEN via environment or Colab secrets. In Colab:
#   from google.colab import userdata
#   os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
HF_TOKEN = os.environ['HF_TOKEN']

login(token=HF_TOKEN, add_to_git_credential=False)
print('logged in as:', whoami()['name'])

MODEL_ID = 'google/medgemma-1.5-4b-it'
print(f'\nchecking access to {MODEL_ID} ...')
try:
    hf_hub_download(repo_id=MODEL_ID, filename='config.json')
    print('  ok - model access granted')
except GatedRepoError:
    print('  BLOCKED - you have not accepted the license.')
    print(f'  visit https://huggingface.co/{MODEL_ID} and click "Agree", then re-run this cell.')
    sys.exit(1)
except RepositoryNotFoundError:
    print('  BLOCKED - repo not found or token lacks Read scope. Regenerate at https://huggingface.co/settings/tokens')
    sys.exit(1)
except Exception as e:
    print(f'  unexpected error: {e}'); raise


## 4. Download & extract the Zenodo dataset (~1.6 GB)

In [ ]:
import os, zipfile, time, requests
from tqdm.auto import tqdm

URL  = 'https://zenodo.org/records/14827784/files/Dataset.zip?download=1'
ZIP  = 'Dataset.zip'
ROOT = 'Dataset'

def download_with_progress(url, dest):
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(dest, 'wb') as f, tqdm(
            total=total, unit='B', unit_scale=True, unit_divisor=1024,
            desc=os.path.basename(dest)
        ) as bar:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
                bar.update(len(chunk))

def extract_with_progress(zip_path, dest_dir):
    with zipfile.ZipFile(zip_path) as z:
        members = z.infolist()
        for m in tqdm(members, desc='extracting', unit='file'):
            z.extract(m, dest_dir)

if not os.path.exists(ROOT):
    if not os.path.exists(ZIP):
        print('downloading Dataset.zip (~1.6 GB) ...')
        t0 = time.time()
        download_with_progress(URL, ZIP)
        print(f'  downloaded in {time.time()-t0:.0f}s')
    print('extracting ...')
    t0 = time.time()
    extract_with_progress(ZIP, '.')
    print(f'  extracted in {time.time()-t0:.0f}s. top-level:', sorted(os.listdir('.'))[:10])
else:
    print(f'{ROOT}/ already present, skipping download')

!ls Dataset/ | head
!find Dataset -maxdepth 4 -type d | head -20


## 5. Build the patient-safe, occlusal-only YOLO split

Same split logic as the YOLO notebook — patient IDs from `anonymous_XXX_XXX_XXX_...`, splits at the patient level, Mandibular + Maxillary_Occlusal views only.


In [ ]:
import re, random, shutil, yaml
from pathlib import Path
from collections import defaultdict, Counter

ROOT = Path('Dataset')
OUT = Path('yolo_split')
VIEWS = ('Mandibular', 'Maxillary_Occlusal')
PATIENT_RE = re.compile(r'anonymous[_-](\d+[_-]\d+[_-]\d+)')
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}
random.seed(42)

def in_view(p): return any(v in str(p) for v in VIEWS)

images, labels = {}, {}
for p in ROOT.rglob('*'):
    if not in_view(p): continue
    s = p.suffix.lower()
    if s in IMG_EXT: images.setdefault(p.stem, p)
    elif s == '.txt' and p.name.lower() not in {'classes.txt','readme.txt'}:
        labels.setdefault(p.stem, p)

pairs = [(images[k], labels.get(k)) for k in images]
groups = defaultdict(list)
for i, (img, _) in enumerate(pairs):
    m = PATIENT_RE.search(img.stem)
    groups[m.group(1) if m else img.stem].append(i)
print(f'occlusal images: {len(pairs)}   patients: {len(groups)}')

gkeys = list(groups.keys()); random.shuffle(gkeys)
n = len(gkeys); tr_cut, va_cut = int(n*0.7), int(n*0.85)
split = {}
for gi, k in enumerate(gkeys):
    s = 'train' if gi < tr_cut else ('val' if gi < va_cut else 'test')
    for idx in groups[k]: split[idx] = s

for sub in ('train','val','test'):
    (OUT/'images'/sub).mkdir(parents=True, exist_ok=True)
    (OUT/'labels'/sub).mkdir(parents=True, exist_ok=True)

counts = Counter()
for i, (img, lbl) in enumerate(pairs):
    s = split[i]; counts[s] += 1
    shutil.copy2(img, OUT/'images'/s/img.name)
    dst = OUT/'labels'/s/(img.stem+'.txt')
    if lbl: shutil.copy2(lbl, dst)
    else:   dst.write_text('')

(OUT/'data.yaml').write_text(yaml.safe_dump({
    'path': str(OUT.resolve()),
    'train': 'images/train', 'val': 'images/val', 'test': 'images/test',
    'nc': 2, 'names': {0: 'd', 1: 'D'},
}, sort_keys=False))
print('split counts:', dict(counts))


## 6. Convert to MedGemma instruction JSONL

Writes `build_vlm_data.py` verbatim from the local version and runs it against the split.


In [ ]:
%%writefile build_vlm_data.py
#!/usr/bin/env python3
"""
build_vlm_data.py - turn the YOLO split into MedGemma instruction data.

READ THIS, it's the crux of your whole "with proper reasoning" request:

  The dataset has boxes and class labels. It has ZERO reasoning text. That text
  has to come from somewhere, and where it comes from decides whether the model
  learns to think or learns to bluff.

  The WRONG way: hand each labelled image to a big VLM and let it write a
  clinical-sounding justification. That model is told the answer in advance, so
  it learns to rationalize whatever label is present - exactly the burn-marks
  failure you already caught (it boxed a finding, then couldn't see the finding
  in its own crop). Synthetic free-text CoT bakes that in and makes it fluent.

  What this script does instead: builds SHORT reasoning traces that are strictly
  DERIVED FROM THE GROUND-TRUTH GEOMETRY - how many lesions, where in the frame,
  how large. Nothing is invented. It's a scaffold, not clinical reasoning.

  The RIGHT way, if you can afford it: have a dentist write even 200-500 real
  rationales. That beats 6,000 templated ones. Use this to bootstrap, then
  replace the traces with clinician text as you get it.

Output: JSONL, one example per line, in the exact [y0,x0,y1,x1]/1000 protocol
your main.py already uses - so inference code doesn't change after fine-tuning.

    python build_vlm_data.py --split ./yolo_split --out ./vlm_data

Requires: pillow, pyyaml
"""

import argparse
import json
import random
from pathlib import Path

import yaml
from PIL import Image, ImageOps

# Mirrors main.py's localization prompt (CXR laterality line removed). Keep it
# identical to what you'll use at inference.
INSTRUCTION = (
    "Instructions:\nThe following user query will require outputting bounding "
    "boxes. The format of bounding boxes coordinates is [y0, x0, y1, x1] where "
    "(y0, x0) is the top-left corner and (y1, x1) the bottom-right corner, with "
    "x0 < x1 and y0 < y1. Normalize all coordinates to the range [0, 1000]. "
    "Output a single parseable json list of objects, each with a \"box_2d\" and "
    "a \"label\" key.\n\nQuery:\nIdentify any carious lesions (tooth decay) "
    "visible in this intraoral photograph. Reason briefly about what you see, "
    "then give the final answer as \"Final Answer: X\" where X is the JSON list."
)


def region_phrase(cx, cy):
    """IMAGE-relative descriptor. NOT clinical tooth numbering, NOT patient
    laterality - a photo can't establish those. Kept deliberately geometric."""
    vert = "upper" if cy < 0.45 else ("lower" if cy > 0.55 else "mid")
    horiz = ("left" if cx < 0.4 else ("right" if cx > 0.6 else "central"))
    return f"{vert} {horiz} region of the image"


def to_1000(cx, cy, w, h):
    x0 = max(0, min(1000, round((cx - w / 2) * 1000)))
    y0 = max(0, min(1000, round((cy - h / 2) * 1000)))
    x1 = max(0, min(1000, round((cx + w / 2) * 1000)))
    y1 = max(0, min(1000, round((cy + h / 2) * 1000)))
    return [y0, x0, y1, x1]


# Single-character dataset labels ("d", "D") are poor LM training targets.
# Map to phrasing the VLM can actually reason about.
LABEL_MAP = {"d": "primary_caries", "D": "permanent_caries"}


def build_response(boxes, names):
    """Grounded trace + JSON answer. Every clause traces to a real annotation."""
    if not boxes:
        # NEGATIVE example. These are non-negotiable - without them the model
        # has no "nothing here" path and will hallucinate a lesion every time,
        # which is precisely the behaviour you want to train OUT.
        trace = ("Scanning the visible dentition surface by surface. The enamel "
                 "appears intact with no discoloration, cavitation, or shadowing "
                 "that would indicate decay.")
        return f"{trace}\nFinal Answer: []"

    n = len(boxes)
    locs = []
    objs = []
    for (c, cx, cy, w, h) in boxes:
        raw = names[c] if c < len(names) else f"class_{c}"
        label = LABEL_MAP.get(raw, raw)
        locs.append(f"a {label.replace('_', ' ')} lesion in the {region_phrase(cx, cy)}")
        objs.append({"box_2d": to_1000(cx, cy, w, h), "label": label})
    lead = "one region" if n == 1 else f"{n} regions"
    trace = (f"Scanning the visible dentition. I can identify {lead} of concern: "
             + "; ".join(locs) + ". Marking the affected area"
             + ("." if n == 1 else "s."))
    answer = json.dumps(objs)
    return f"{trace}\nFinal Answer: ```json{answer}```"


def parse_label(lbl: Path):
    if lbl is None or not lbl.exists():
        return []
    out = []
    for line in lbl.read_text().splitlines():
        p = line.split()
        if len(p) >= 5:
            out.append((int(float(p[0])), float(p[1]), float(p[2]),
                        float(p[3]), float(p[4])))
    return out


def build_split(split_dir: Path, names, sub: str, out_path: Path, keep_size: int):
    img_dir = split_dir / "images" / sub
    lbl_dir = split_dir / "labels" / sub
    n_written = n_neg = 0
    with out_path.open("w") as f:
        for img in sorted(img_dir.iterdir()):
            if img.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}:
                continue
            boxes = parse_label(lbl_dir / (img.stem + ".txt"))
            if not boxes:
                n_neg += 1
            # optionally normalise orientation + downscale a copy for training
            im = ImageOps.exif_transpose(Image.open(img)).convert("RGB")
            if keep_size and max(im.size) > keep_size:
                im.thumbnail((keep_size, keep_size))
            saved = out_path.parent / "images" / sub / (img.stem + ".png")
            saved.parent.mkdir(parents=True, exist_ok=True)
            im.save(saved)

            record = {
                "image": str(saved.resolve()),
                "messages": [
                    {"role": "user", "content": INSTRUCTION},
                    {"role": "assistant", "content": build_response(boxes, names)},
                ],
            }
            f.write(json.dumps(record) + "\n")
            n_written += 1
    print(f"{sub}: {n_written} examples ({n_neg} negatives) -> {out_path}")
    if n_neg == 0 and sub == "train":
        print("  !! ZERO negatives in train. The model will learn to always find")
        print("     something. Source lesion-free intraoral images before training.")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--split", required=True, help="yolo_split dir from prepare_data.py")
    ap.add_argument("--out", default="./vlm_data")
    ap.add_argument("--keep-size", type=int, default=1024,
                    help="downscale longest side to this for training copies; 0 = keep original")
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    random.seed(args.seed)
    split_dir = Path(args.split)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    names_map = yaml.safe_load((split_dir / "data.yaml").read_text())["names"]
    names = [names_map[i] for i in sorted(names_map)]
    print(f"classes: {names}")

    for sub in ("train", "val", "test"):
        build_split(split_dir, names, sub, out / f"{sub}.jsonl", args.keep_size)

    print("\nSanity-check a few assistant messages by eye before training.")
    print("If the templated traces read as fake to you, they'll read as fake to")
    print("the model too - that's your cue to get real clinician rationales.")


if __name__ == "__main__":
    main()


In [ ]:
!python build_vlm_data.py --split ./yolo_split --out ./vlm_data


## 7. LoRA fine-tune MedGemma-4B

Same as `finetune_medgemma.py` locally. Defaults to **4-bit QLoRA** (fits A100-40GB with headroom). Remove `--four-bit` and lower `--grad-accum` if you have A100-80GB.


In [ ]:
%%writefile finetune_medgemma.py
#!/usr/bin/env python3
"""
finetune_medgemma.py - LoRA fine-tune MedGemma 1.5 4B on your caries JSONL.

REALITY CHECK ON HARDWARE:
  Training a 4B multimodal model with a vision tower does NOT fit comfortably in
  16 GB. On your Mac mini this will either OOM or crawl. Two honest options:
    (A) Rent one A100/H100 for a few hours (~cost of lunch) and run THIS script
        with --device cuda. Recommended. Finishes overnight.
    (B) Stay local via Apple's MLX, which handles unified memory better:
            pip install mlx-vlm
            python -m mlx_vlm.lora \
                --model mlx-community/medgemma-1.5-4b-it-bf16 \
                --train data/train.jsonl --batch-size 1 --iters 2000
        (flag names drift between mlx-vlm versions - check `python -m mlx_vlm.lora -h`)

  This script is the CUDA/transformers path (A). It's the robust one to build on.

VERSION SENSITIVITY: the multimodal SFT surface in transformers changes often.
The lines most likely to need adjusting against your installed versions are
tagged  # >>> VERIFY. Check the current MedGemma model card and transformers
docs if any of them throw.

    python finetune_medgemma.py --data ./vlm_data --device cuda

Requires: transformers, peft, accelerate, bitsandbytes (for 4-bit), pillow, torch
"""

import argparse
import json
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import Dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,   # >>> VERIFY: generic loader for Gemma3-family VLMs
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)

MODEL_ID = "google/medgemma-1.5-4b-it"   # gated - `huggingface-cli login` first


class CariesVLM(Dataset):
    def __init__(self, jsonl):
        self.rows = [json.loads(l) for l in Path(jsonl).read_text().splitlines() if l.strip()]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return self.rows[i]


def make_collator(processor):
    def collate(batch):
        images, texts = [], []
        for row in batch:
            img = Image.open(row["image"]).convert("RGB")
            # Build the chat with an image placeholder in the user turn.
            msgs = [
                {"role": "user", "content": [
                    {"type": "image"},
                    {"type": "text", "text": row["messages"][0]["content"]},
                ]},
                {"role": "assistant", "content": [
                    {"type": "text", "text": row["messages"][1]["content"]},
                ]},
            ]
            # >>> VERIFY: apply_chat_template signature / add_generation_prompt behaviour
            text = processor.apply_chat_template(msgs, tokenize=False,
                                                 add_generation_prompt=False)
            images.append([img])
            texts.append(text.strip())

        batch_enc = processor(
            text=texts, images=images,
            return_tensors="pt", padding=True,
        )
        labels = batch_enc["input_ids"].clone()

        # Mask pad tokens.
        labels[labels == processor.tokenizer.pad_token_id] = -100
        # Mask image-token positions so loss is computed on text only.
        # >>> VERIFY: image_token_id attribute name varies (image_token_index etc.)
        img_tok = getattr(processor.tokenizer, "image_token_id", None)
        if img_tok is not None:
            labels[labels == img_tok] = -100
        batch_enc["labels"] = labels
        return batch_enc
    return collate


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="vlm_data dir with train/val jsonl")
    ap.add_argument("--out", default="./medgemma_caries_lora")
    ap.add_argument("--device", default="cuda", choices=["cuda", "cpu"])
    ap.add_argument("--epochs", type=float, default=3)
    ap.add_argument("--batch", type=int, default=1)
    ap.add_argument("--grad-accum", type=int, default=8)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--four-bit", action="store_true",
                    help="QLoRA - load base in 4-bit to fit smaller GPUs")
    args = ap.parse_args()

    processor = AutoProcessor.from_pretrained(MODEL_ID)

    quant = None
    if args.four_bit:
        quant = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=quant,
        torch_dtype=torch.bfloat16,
        device_map="auto" if args.device == "cuda" else None,
        attn_implementation="eager",   # >>> VERIFY: Gemma3 recommends eager attn
    )

    # LoRA on the LANGUAGE tower only. Freeze the vision encoder - you don't have
    # remotely enough data to retrain how it sees, and touching it destabilizes
    # everything. This is the single most important choice in the file.
    lora = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        # >>> VERIFY: exclude vision-tower module names if the above regex-matches them
    )
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()

    train_ds = CariesVLM(Path(args.data) / "train.jsonl")
    val_ds = CariesVLM(Path(args.data) / "val.jsonl")
    collate = make_collator(processor)

    targs = TrainingArguments(
        output_dir=args.out,
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        bf16=True,
        gradient_checkpointing=True,
        logging_steps=10,
        eval_strategy="steps", eval_steps=100,
        save_strategy="steps", save_steps=200, save_total_limit=2,
        report_to="none",
        remove_unused_columns=False,   # >>> critical: keeps our dict columns intact
        dataloader_num_workers=4,
    )

    trainer = Trainer(
        model=model, args=targs,
        train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=collate,
    )
    trainer.train()

    trainer.save_model(args.out)
    processor.save_pretrained(args.out)
    print(f"\nLoRA adapter -> {args.out}")
    print("Inference: load base MedGemma + this adapter, then reuse your main.py")
    print("prompt/parsing unchanged. FIRST evaluate on test.jsonl and compare")
    print("mAP against the YOLO baseline before believing any of this helped.")


if __name__ == "__main__":
    main()


In [ ]:
# ~60-90 min on A100-40GB. Watch loss in output.
# Reduce epochs to 1 first if you just want to smoke-test the pipeline.
!python finetune_medgemma.py \
    --data ./vlm_data \
    --device cuda \
    --four-bit \
    --epochs 3 \
    --batch 1 \
    --grad-accum 8 \
    --lr 1e-4 \
    --out ./medgemma_caries_lora


## 8. Quick smoke test — run the fine-tuned model on 3 test examples

In [ ]:
import json, torch
from pathlib import Path
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = 'google/medgemma-1.5-4b-it'
ADAPTER  = './medgemma_caries_lora'

processor = AutoProcessor.from_pretrained(MODEL_ID)
qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                          bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
base = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, quantization_config=qcfg, torch_dtype=torch.bfloat16,
    device_map='auto', attn_implementation='eager')
base.eval()

# LoRA on top of the same base weights
tuned = PeftModel.from_pretrained(base, ADAPTER)
tuned.eval()

def generate(model, image, prompt_text):
    msgs = [{'role': 'user', 'content': [
        {'type': 'image'}, {'type': 'text', 'text': prompt_text}]}]
    prompt = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[[image]], return_tensors='pt').to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    return processor.tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

test_rows = [json.loads(l) for l in Path('vlm_data/test.jsonl').read_text().splitlines()][:3]
for i, row in enumerate(test_rows):
    img = Image.open(row['image']).convert('RGB')
    user_prompt = row['messages'][0]['content']
    print(f'=== test [{i}]  {Path(row["image"]).name} ===')
    print('GROUND TRUTH  :', row['messages'][1]['content'][:300], '...')
    print('BASE MODEL    :', generate(base,  img, user_prompt)[:300], '...')
    print('FINE-TUNED    :', generate(tuned, img, user_prompt)[:300], '...')
    print()

# If BASE and FINE-TUNED look nearly identical, the fine-tune didn't stick.
# Likely causes: lr too low, epochs too few, LoRA rank too low, or bad target_modules.


## 9. Zip & download the LoRA adapter

In [ ]:
import shutil
shutil.make_archive('medgemma_caries_lora', 'zip', '.', 'medgemma_caries_lora')
!ls -lh medgemma_caries_lora.zip
from google.colab import files
files.download('medgemma_caries_lora.zip')
